# PDF 页面纠偏矫正实验

对 `../formal/荆霄鹏规范汉字行楷7000字.pdf` 使用三种工具进行页面 deskew（纠偏）处理，对比效果：

| 工具 | 原理 | 输入/输出 |
|------|------|----------|
| **OCRmyPDF** | 内部调用 Leptonica 做旋转检测 + deskew | PDF → PDF |
| **unpaper** | 基于投影直方图的倾斜检测 + 仿射变换 | PNM 图片 → PNM 图片 |
| **ScanTailor Advanced** | 综合自动 deskew + content detection | 图片 → 图片 |

## 0. 安装依赖

In [ ]:
# %%bash

# # Python 库
# uv pip install -q ocrmypdf PyMuPDF pdf2image img2pdf

# # 系统工具
# apt-get update -qq
# apt-get install -y -qq poppler-utils unpaper ghostscript tesseract-ocr > /dev/null 2>&1

# echo '--- 版本信息 ---'
# ocrmypdf --version
# unpaper --version
# echo 'poppler-utils:' && pdftoppm -v 2>&1 || true

In [ ]:
# %%bash

# # Python 库
# uv pip install -q ocrmypdf PyMuPDF pdf2image img2pdf

# # 系统工具
# apt-get update -qq
# apt-get install -y -qq poppler-utils unpaper ghostscript tesseract-ocr > /dev/null 2>&1

# echo '--- 版本信息 ---'
# ocrmypdf --version
# unpaper --version
# echo 'poppler-utils:' && pdftoppm -v 2>&1 || true

In [ ]:
# %%bash
# # ScanTailor Advanced — 从官方 PPA 或 GitHub release 安装
# # 如果 apt 源中没有，尝试从 GitHub 下载 AppImage

# if command -v scantailor-cli &>/dev/null; then
#     echo "scantailor-cli 已安装"
#     scantailor-cli --help 2>&1 | head -5
#     exit 0
# fi

# echo '尝试 apt 安装 scantailor / scantailor-advanced ...'
# apt-get install -y -qq scantailor-advanced 2>/dev/null \
#   || apt-get install -y -qq scantailor 2>/dev/null \
#   || {
#     echo '=== apt 安装失败，尝试从源码编译 ScanTailor Advanced ==='
#     apt-get install -y -qq \
#       build-essential cmake libjpeg-dev libpng-dev libtiff-dev \
#       zlib1g-dev libboost-all-dev qtbase5-dev libqt5opengl5-dev \
#       > /dev/null 2>&1

#     WORKDIR=$(mktemp -d)
#     cd "$WORKDIR"
#     git clone --depth 1 https://github.com/4lex4/scantailor-advanced.git
#     cd scantailor-advanced
#     mkdir build && cd build
#     cmake .. -DCMAKE_BUILD_TYPE=Release -DCMAKE_INSTALL_PREFIX=/usr/local 2>&1 | tail -3
#     make -j$(nproc) 2>&1 | tail -3
#     make install
#     cd /
#     rm -rf "$WORKDIR"
#     echo 'ScanTailor Advanced 编译安装完成'
# }

# # 确认 CLI 可用
# which scantailor-cli && scantailor-cli --help 2>&1 | head -5 || echo '警告: scantailor-cli 未找到，后续实验将跳过'

## 1. 配置与辅助函数

In [3]:
import os, shutil, subprocess, tempfile, math
from pathlib import Path

import fitz                       # PyMuPDF
from PIL import Image
from pdf2image import convert_from_path
from IPython.display import display, HTML
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

PDF_PATH = Path("../formal/荆霄鹏规范汉字行楷7000字.pdf")
OUTPUT_DIR = Path("deskew_output")
OUTPUT_DIR.mkdir(exist_ok=True)

SAMPLE_PAGES = [10, 12, 13, 14, 15,16]  # 选取有歪斜的页面做实验，可按需修改
DPI = 300

HAS_SCANTAILOR = shutil.which("scantailor-cli") is not None
print(f"PDF: {PDF_PATH}  (exists={PDF_PATH.exists()})")
print(f"Sample pages: {SAMPLE_PAGES}")
print(f"ScanTailor CLI available: {HAS_SCANTAILOR}")

PDF: ../formal/荆霄鹏规范汉字行楷7000字.pdf  (exists=True)
Sample pages: [10, 12, 13, 14, 15, 16]
ScanTailor CLI available: False


In [4]:
def pdf_page_to_png(pdf_path: Path, page_idx: int, out_path: Path, dpi: int = DPI):
    """用 PyMuPDF 将单页渲染为 PNG。"""
    doc = fitz.open(str(pdf_path))
    page = doc[page_idx]
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat)
    pix.save(str(out_path))
    doc.close()
    return out_path


def show_comparison(images: dict, title: str = "", max_width: int = 350):
    """并排显示多张图片用于对比。images = {label: path_or_PIL}"""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 8))
    if n == 1:
        axes = [axes]
    for ax, (label, img) in zip(axes, images.items()):
        if isinstance(img, (str, Path)):
            img = Image.open(img)
        ax.imshow(img)
        ax.set_title(label, fontsize=11)
        ax.axis("off")
    if title:
        fig.suptitle(title, fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    plt.close(fig)

## 2. 提取样本页面原始图片

In [5]:
orig_dir = OUTPUT_DIR / "original"
orig_dir.mkdir(exist_ok=True)

orig_images = {}
for pi in SAMPLE_PAGES:
    out = orig_dir / f"page_{pi:03d}.png"
    pdf_page_to_png(PDF_PATH, pi, out)
    orig_images[pi] = out
    print(f"Page {pi}: {out}  ({Image.open(out).size})")

print(f"\n共提取 {len(orig_images)} 页原始图片")

Page 10: deskew_output/original/page_010.png  ((4500, 4821))
Page 12: deskew_output/original/page_012.png  ((4500, 4813))
Page 13: deskew_output/original/page_013.png  ((4500, 4813))
Page 14: deskew_output/original/page_014.png  ((4500, 4842))
Page 15: deskew_output/original/page_015.png  ((4500, 4863))
Page 16: deskew_output/original/page_016.png  ((4500, 4855))

共提取 6 页原始图片


---
## 3. 实验一：OCRmyPDF `--deskew`

OCRmyPDF 内部使用 Leptonica 库进行旋转角度检测并校正。
- 优点：直接 PDF→PDF，一行命令
- 可用参数：`--deskew`（启用纠偏），`--deskew-threshold`（角度阈值，默认不需要设）

In [6]:
ocr_out_dir = OUTPUT_DIR / "ocrmypdf"
ocr_out_dir.mkdir(exist_ok=True)
ocr_output_pdf = ocr_out_dir / "deskewed.pdf"

cmd = [
    "ocrmypdf",
    "--deskew",              # 启用纠偏
    "--skip-text",           # 跳过 OCR 文字层（这里只关心纠偏）
    "--output-type", "pdf",
    "--pages", ",".join(str(p + 1) for p in SAMPLE_PAGES),  # 1-indexed
    str(PDF_PATH),
    str(ocr_output_pdf),
]
print("命令:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=True, text=True)
print("STDOUT:", result.stdout[-500:] if result.stdout else "(empty)")
print("STDERR:", result.stderr[-1000:] if result.stderr else "(empty)")
print(f"返回码: {result.returncode}")
print(f"输出文件: {ocr_output_pdf}  (exists={ocr_output_pdf.exists()})")

命令: ocrmypdf --deskew --skip-text --output-type pdf --pages 11,13,14,15,16,17 ../formal/荆霄鹏规范汉字行楷7000字.pdf deskew_output/ocrmypdf/deskewed.pdf


STDOUT: (empty)
STDERR: Start processing 12 pages concurrently
   16 [tesseract] lots of diacritics - possibly poor OCR
   15 [tesseract] lots of diacritics - possibly poor OCR
   11 [tesseract] lots of diacritics - possibly poor OCR
   17 [tesseract] lots of diacritics - possibly poor OCR
   14 [tesseract] lots of diacritics - possibly poor OCR
   13 [tesseract] lots of diacritics - possibly poor OCR
Postprocessing...
Image optimization ratio: 1.02 savings: 2.2%
Total file size ratio: 0.99 savings: -0.8%

返回码: 0
输出文件: deskew_output/ocrmypdf/deskewed.pdf  (exists=True)


In [7]:
ocr_images = {}
if ocr_output_pdf.exists():
    doc = fitz.open(str(ocr_output_pdf))
    for i, pi in enumerate(SAMPLE_PAGES):
        if i >= doc.page_count:
            break
        out = ocr_out_dir / f"page_{pi:03d}.png"
        page = doc[i]
        mat = fitz.Matrix(DPI / 72, DPI / 72)
        pix = page.get_pixmap(matrix=mat)
        pix.save(str(out))
        ocr_images[pi] = out
    doc.close()
    print(f"OCRmyPDF 纠偏结果提取了 {len(ocr_images)} 页")
else:
    print("OCRmyPDF 输出文件不存在，请检查上一步错误")

OCRmyPDF 纠偏结果提取了 6 页


---
## 4. 实验二：unpaper

unpaper 是一个专门用于扫描件后处理的工具，支持 deskew、border removal、noise reduction 等。
- 输入：PBM/PGM/PPM 格式
- 需要先把 PNG 转成 PNM，处理后再转回 PNG

In [8]:
unpaper_dir = OUTPUT_DIR / "unpaper"
unpaper_dir.mkdir(exist_ok=True)

unpaper_images = {}
for pi in SAMPLE_PAGES:
    src_png = orig_images[pi]
    # unpaper 需要 PNM 格式输入
    pnm_in = unpaper_dir / f"page_{pi:03d}_in.pnm"
    pnm_out = unpaper_dir / f"page_{pi:03d}_out.pnm"
    dst_png = unpaper_dir / f"page_{pi:03d}.png"

    # PNG → PNM
    Image.open(src_png).save(str(pnm_in))

    cmd = [
        "unpaper",
        "--no-mask-scan",       # 不做 mask 检测，加快速度
        "--no-border-scan",     # 不做边框检测
        "--no-noisefilter",     # 不做降噪（只看纠偏效果）
        "--no-blurfilter",
        "--no-grayfilter",
        "--dpi", str(DPI),
        str(pnm_in),
        str(pnm_out),
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"Page {pi} unpaper 失败: {result.stderr[:300]}")
        continue

    # PNM → PNG
    Image.open(str(pnm_out)).save(str(dst_png))
    unpaper_images[pi] = dst_png

    # 清理中间 PNM 文件（较大）
    pnm_in.unlink(missing_ok=True)
    pnm_out.unlink(missing_ok=True)

    print(f"Page {pi}: done")

print(f"\nunpaper 处理完成 {len(unpaper_images)} 页")

Page 10: done
Page 12: done
Page 13: done
Page 14: done
Page 15: done
Page 16: done

unpaper 处理完成 6 页


---
## 5. 实验三：ScanTailor Advanced

ScanTailor Advanced 的 CLI 模式可以自动完成 deskew、content detection、page splitting 等操作。
- `scantailor-cli` 接受一组图片，输出处理后的 TIFF
- 如果编译安装失败，本节将跳过

In [ ]:
st_dir = OUTPUT_DIR / "scantailor"
st_dir.mkdir(exist_ok=True)
st_images = {}

if not HAS_SCANTAILOR:
    print("⚠ scantailor-cli 不可用，跳过本实验。")
    print("  可手动运行上面的编译安装 Cell 后重新执行。")
else:
    st_input_dir = st_dir / "input"
    st_output_dir = st_dir / "output"
    st_input_dir.mkdir(exist_ok=True)
    st_output_dir.mkdir(exist_ok=True)

    # 准备输入图片（TIFF 格式效果最佳）
    input_files = []
    for pi in SAMPLE_PAGES:
        src = orig_images[pi]
        dst = st_input_dir / f"page_{pi:03d}.tif"
        Image.open(src).save(str(dst), compression="tiff_deflate")
        input_files.append(str(dst))

    cmd = [
        "scantailor-cli",
        "--deskew=auto",
        "--content-detection=cautious",
        "--output-dpi", str(DPI),
        *input_files,
        str(st_output_dir),
    ]
    print("命令:", " ".join(cmd[:6]), "...", cmd[-1])
    result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)
    print("STDOUT:", result.stdout[-500:] if result.stdout else "(empty)")
    print("STDERR:", result.stderr[-500:] if result.stderr else "(empty)")
    print(f"返回码: {result.returncode}")

    for pi in SAMPLE_PAGES:
        tif_out = st_output_dir / f"page_{pi:03d}.tif"
        png_out = st_dir / f"page_{pi:03d}.png"
        if tif_out.exists():
            Image.open(str(tif_out)).convert("RGB").save(str(png_out))
            st_images[pi] = png_out

    print(f"ScanTailor 处理完成 {len(st_images)} 页")

---
## 6. 效果对比

In [ ]:
for pi in SAMPLE_PAGES:
    panels = {"原始": orig_images.get(pi)}
    if pi in ocr_images:
        panels["OCRmyPDF"] = ocr_images[pi]
    if pi in unpaper_images:
        panels["unpaper"] = unpaper_images[pi]
    if pi in st_images:
        panels["ScanTailor"] = st_images[pi]

    show_comparison(panels, title=f"第 {pi} 页 纠偏对比")

## 7. 倾斜角度量化分析（可选）

使用 OpenCV 的霍夫线变换估算各版本的残余倾斜角度。

In [ ]:
try:
    import cv2
    import numpy as np

    def estimate_skew_angle(img_path: str) -> float:
        """通过霍夫线变换估算图片倾斜角度（度）。"""
        img = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
        if img is None:
            return float('nan')
        edges = cv2.Canny(img, 50, 150, apertureSize=3)
        lines = cv2.HoughLinesP(edges, 1, math.pi / 180, threshold=200,
                                minLineLength=img.shape[1] // 4, maxLineGap=20)
        if lines is None:
            return 0.0
        angles = []
        for line in lines:
            x1, y1, x2, y2 = line[0]
            angle = math.degrees(math.atan2(y2 - y1, x2 - x1))
            # 只取近似水平的线
            if abs(angle) < 30:
                angles.append(angle)
        return float(np.median(angles)) if angles else 0.0

    print(f"{'Page':<6} {'原始':>8} {'OCRmyPDF':>10} {'unpaper':>10} {'ScanTailor':>12}")
    print("-" * 50)
    for pi in SAMPLE_PAGES:
        row = f"{pi:<6}"
        row += f" {estimate_skew_angle(orig_images.get(pi, '')):>7.2f}°"
        row += f" {estimate_skew_angle(ocr_images.get(pi, '')):>9.2f}°"
        row += f" {estimate_skew_angle(unpaper_images.get(pi, '')):>9.2f}°"
        if st_images:
            row += f" {estimate_skew_angle(st_images.get(pi, '')):>11.2f}°"
        else:
            row += f" {'N/A':>12}"
        print(row)

except ImportError:
    print("cv2 未安装，跳过角度分析。可运行: pip install opencv-python-headless")

---
## 8. 全量处理（可选）

如果对样本页面效果满意，可以取消注释下方代码对整个 PDF 执行纠偏。

In [ ]:
# === OCRmyPDF 全量处理（最简单） ===
# full_output = OUTPUT_DIR / "full_deskewed_ocrmypdf.pdf"
# !ocrmypdf --deskew --skip-text "{PDF_PATH}" "{full_output}"
# print(f"输出: {full_output}")

In [ ]:
# === unpaper 全量处理 ===
# import img2pdf
#
# doc = fitz.open(str(PDF_PATH))
# all_png_paths = []
# for pi in range(doc.page_count):
#     tmp_pnm_in = OUTPUT_DIR / f"tmp_in_{pi:04d}.pnm"
#     tmp_pnm_out = OUTPUT_DIR / f"tmp_out_{pi:04d}.pnm"
#     out_png = OUTPUT_DIR / f"unpaper_full_{pi:04d}.png"
#
#     pdf_page_to_png(PDF_PATH, pi, OUTPUT_DIR / f"tmp_{pi:04d}.png")
#     Image.open(OUTPUT_DIR / f"tmp_{pi:04d}.png").save(str(tmp_pnm_in))
#
#     subprocess.run(["unpaper", "--dpi", str(DPI),
#                     str(tmp_pnm_in), str(tmp_pnm_out)],
#                    capture_output=True)
#     Image.open(str(tmp_pnm_out)).save(str(out_png))
#     all_png_paths.append(str(out_png))
#     tmp_pnm_in.unlink(missing_ok=True)
#     tmp_pnm_out.unlink(missing_ok=True)
#     (OUTPUT_DIR / f"tmp_{pi:04d}.png").unlink(missing_ok=True)
# doc.close()
#
# # 合并为 PDF
# pdf_bytes = img2pdf.convert(all_png_paths)
# (OUTPUT_DIR / "full_deskewed_unpaper.pdf").write_bytes(pdf_bytes)
# print(f"输出: {OUTPUT_DIR / 'full_deskewed_unpaper.pdf'}")

---
## 9. 小结

| 工具 | 易用性 | 纠偏效果 | 速度 | 备注 |
|------|--------|----------|------|------|
| OCRmyPDF | ★★★★★ | 通常较好 | 中等 | PDF→PDF 直出，最省心 |
| unpaper | ★★★☆☆ | 取决于内容 | 较快 | 需要 PNM 中转，参数可调性强 |
| ScanTailor | ★★☆☆☆ | 通常最佳 | 较慢 | 编译安装较麻烦，但综合处理能力最强 |